## Install dependencies

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes sentencepiece pypdf \
    sentence-transformers google-generativeai tqdm


## Imports & paths

In [ ]:
import os, re, json, random, time
from pathlib import Path
import torch
import numpy as np

random.seed(42)

LECTURES_DIR = "/kaggle/input/lectures"
OUTPUT_DIR   = "/kaggle/working"

assert os.path.isdir(LECTURES_DIR), (
    f"'{LECTURES_DIR}' not found. Attach lecture PDFs and update LECTURES_DIR."
)

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("Paths OK ")


## Extract text & semantic chunk all courses

In [ ]:
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

def extract_pdf(path):
    reader = PdfReader(path)
    text = ""
    for page in reader.pages:
        t = page.extract_text()
        if t:
            text += t + " "
    return text

def clean_text(text):
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f]', ' ', text)
    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

embedder = SentenceTransformer(
    "all-MiniLM-L6-v2",
    device="cuda" if torch.cuda.is_available() else "cpu"
)

SENTENCE_SPLIT_RE = re.compile(r'(?<=[.!?])\s+(?=[A-Z0-9])')

def split_sentences(text):
    return [s.strip() for s in SENTENCE_SPLIT_RE.split(text) if s.strip()]

def semantic_chunk(text, sim_threshold=0.45, min_words=120, max_words=300, overlap_sentences=2):
    sentences = split_sentences(text)
    if len(sentences) < 2:
        return [text] if text.strip() else []
    embeddings = embedder.encode(sentences, batch_size=64, show_progress_bar=False, normalize_embeddings=True)
    chunks, current, cur_words = [], [sentences[0]], len(sentences[0].split())
    for i in range(1, len(sentences)):
        sim = float(np.dot(embeddings[i-1], embeddings[i]))
        sw = len(sentences[i].split())
        if (sim < sim_threshold and cur_words >= min_words) or (cur_words + sw > max_words):
            chunks.append(" ".join(current))
            carry = current[-overlap_sentences:] if overlap_sentences > 0 else []
            current = carry + [sentences[i]]
            cur_words = sum(len(s.split()) for s in current)
        else:
            current.append(sentences[i])
            cur_words += sw
    if current:
        chunks.append(" ".join(current))
    return chunks

courses_raw_text = {}
for course in sorted(os.listdir(LECTURES_DIR)):
    course_path = os.path.join(LECTURES_DIR, course)
    if not os.path.isdir(course_path):
        continue
    print("Reading:", course)
    full_text = ""
    for file in sorted(os.listdir(course_path)):
        if file.lower().endswith(".pdf"):
            full_text += extract_pdf(os.path.join(course_path, file)) + " "
    courses_raw_text[course] = clean_text(full_text)
    print(f"  -> {len(courses_raw_text[course].split())} words")

courses_data = {}
for course, text in courses_raw_text.items():
    chunks = semantic_chunk(text)
    courses_data[course] = chunks
    wc = [len(c.split()) for c in chunks]
    print(f"{course}: {len(chunks)} chunks | avg {sum(wc)/len(wc):.0f} words (min {min(wc)}, max {max(wc)})")

print("\nAll courses chunked \u2705")


## Load Qwen2.5-14B-Instruct 4-bit

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

GEN_MODEL_NAME = "Qwen/Qwen2.5-14B-Instruct"
gen_tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL_NAME)

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

gen_model = AutoModelForCausalLM.from_pretrained(
    GEN_MODEL_NAME,
    device_map="auto",
    quantization_config=quant_config,
    torch_dtype=torch.float16,
)
print("Generator loaded ")


## All shared helpers

In [ ]:
QUESTION_COUNTS = [2, 5, 8]
ALL_MODES = ["mcq", "tf", "mixed"]
PROGRESS_PATH = f"{OUTPUT_DIR}/generation_progress.json"
DATASET_PATH  = f"{OUTPUT_DIR}/quiz_dataset_v2.json"


#Generator
def qwen_generate(prompt, max_new_tokens=700, temperature=0.5):
    messages = [{"role": "user", "content": prompt}]
    text = gen_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = gen_tokenizer(text, return_tensors="pt").to(gen_model.device)
    with torch.no_grad():
        output_ids = gen_model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=True,
            temperature=temperature, top_p=0.9, repetition_penalty=1.15,
            pad_token_id=gen_tokenizer.eos_token_id,
        )
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return gen_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


#Prompt builder
_BASE_RULES = (
    "You are generating university exam questions strictly from the lecture text provided.\n\n"
    "STRICT RULES:\n"
    "- Output ONLY questions, nothing else (no preamble, no explanations, no summary).\n"
    "- Every question and its correct answer MUST be fully supported by the lecture text below.\n"
    "  Do not introduce facts, names, or claims that are not present in the text.\n"
    "- Do NOT copy full sentences verbatim from the lecture text -- rephrase into a question.\n"
    "- Do NOT write the words \"Example\" or \"Question\" as labels.\n"
    "- Do NOT add separators like ---.\n"
    "- Use only Q1, Q2, Q3... labels, sequential, no gaps.\n"
    "- For MCQ: always provide exactly 4 options (A, B, C, D), exactly one correct.\n"
    "- For True/False: Answer must be exactly True or False.\n"
    "- Never output short-answer or open-ended questions.\n"
    "- Generate EXACTLY the number of questions requested. Do not stop early.\n"
    "- If you reach the requested count, STOP -- do not add extra questions after."
)

def _mcq_task(n):
    return (
        f"Generate EXACTLY {n} Multiple Choice Questions.\n\n"
        f"FORMAT EXACTLY (repeat this block {n} times):\n\n"
        "Q1: question text\n"
        "A) option\n"
        "B) option\n"
        "C) option\n"
        "D) option\n"
        "Answer: letter"
    )

def _tf_task(n):
    return (
        f"Generate EXACTLY {n} True/False Questions.\n\n"
        f"FORMAT EXACTLY (repeat this block {n} times):\n\n"
        "Q1: statement\n"
        "Answer: True/False"
    )

def _mixed_task(n):
    n_mcq = max(1, n // 3)
    n_tf = n - n_mcq
    return (
        f"Generate EXACTLY {n} questions total:\n"
        f"- {n_mcq} Multiple Choice Question(s)\n"
        f"- {n_tf} True/False Question(s)\n"
        f"In any order, sequentially numbered Q1..Q{n}.\n\n"
        "FORMAT EXACTLY:\n\n"
        "For MCQ:\n"
        "Qn: question text\n"
        "A) option\n"
        "B) option\n"
        "C) option\n"
        "D) option\n"
        "Answer: letter\n\n"
        "For True/False:\n"
        "Qn: statement\n"
        "Answer: True/False"
    )

def build_prompt(course, chunk, mode, n_questions):
    task = {"mcq": _mcq_task, "tf": _tf_task, "mixed": _mixed_task}[mode](n_questions)
    return (
        f"{_BASE_RULES}\n\n"
        f"Course: {course}\n\n"
        f"Lecture Text:\n{chunk}\n\n"
        f"{task}"
    )


#Validators
def validate_mcq(text, expected_n):
    if len(re.findall(r"Q\d+:", text)) != expected_n:
        return False
    for letter in "ABCD":
        if len(re.findall(rf"\n{letter}\)", text)) != expected_n:
            return False
    return len(re.findall(r"Answer:\s*([A-D])", text)) == expected_n

def validate_tf(text, expected_n):
    if len(re.findall(r"Q\d+:", text)) != expected_n:
        return False
    return len(re.findall(r"Answer:\s*(True|False)", text)) == expected_n

def validate_mixed(text, expected_n):
    if len(re.findall(r"Q\d+:", text)) != expected_n:
        return False
    mcq_ans = re.findall(r"Answer:\s*([A-D])", text)
    tf_ans  = re.findall(r"Answer:\s*(True|False)", text)
    if len(mcq_ans) + len(tf_ans) != expected_n:
        return False
    for letter in "ABCD":
        if len(re.findall(rf"\n{letter}\)", text)) != len(mcq_ans):
            return False
    return True

VALIDATORS = {"mcq": validate_mcq, "tf": validate_tf, "mixed": validate_mixed}


def generate_valid_sample(course, chunk, mode, max_attempts=3):
    n_questions = random.choice(QUESTION_COUNTS)
    for _ in range(max_attempts):
        prompt = build_prompt(course, chunk, mode, n_questions)
        output = qwen_generate(prompt, max_new_tokens=120 + n_questions * 90)
        if VALIDATORS[mode](output, n_questions):
            return output, n_questions
    return None, n_questions


#Progress helpers
def load_progress():
    if os.path.exists(PROGRESS_PATH) and os.path.exists(DATASET_PATH):
        with open(DATASET_PATH, "r", encoding="utf-8") as f:
            dataset = json.load(f)
        with open(PROGRESS_PATH, "r", encoding="utf-8") as f:
            progress = json.load(f)
        done_keys    = set(progress["done_keys"])
        failed_count = progress.get("failed_samples", 0)
        print(f"Resuming: {len(dataset)} samples, {len(done_keys)} combos done.")
        return dataset, done_keys, failed_count
    print("No prior progress -- starting fresh.")
    return [], set(), 0

def save_progress(dataset, done_keys, failed_count):
    with open(DATASET_PATH, "w", encoding="utf-8") as f:
        json.dump(dataset, f, indent=2, ensure_ascii=False)
    with open(PROGRESS_PATH, "w", encoding="utf-8") as f:
        json.dump({"done_keys": list(done_keys), "failed_samples": failed_count}, f, indent=2)


#Main generation loop
from tqdm import tqdm

def run_generation_for_courses(course_chunk_specs):
    final_dataset, done_keys, failed_samples = load_progress()

    total_units = 0
    for spec in course_chunk_specs:
        if len(spec) == 2:
            _, ch = spec
            total_units += len(ch) * len(ALL_MODES)
        else:
            _, ch, s, e = spec
            total_units += len(ch[s:e]) * len(ALL_MODES)

    with tqdm(total=total_units, desc="Generating") as pbar:
        for spec in course_chunk_specs:
            if len(spec) == 2:
                course, all_chunks = spec
                sub_chunks, offset = all_chunks, 0
            else:
                course, all_chunks, start, end = spec
                sub_chunks, offset = all_chunks[start:end], start

            for rel_idx, chunk in enumerate(sub_chunks):
                abs_idx = offset + rel_idx
                for mode in ALL_MODES:
                    key = f"{course}||{abs_idx}||{mode}"
                    if key in done_keys:
                        pbar.update(1)
                        continue

                    output, n_questions = generate_valid_sample(course, chunk, mode)

                    if output is None:
                        failed_samples += 1
                    else:
                        final_dataset.append({
                            "instruction": f"Generate {n_questions} {mode} questions",
                            "course": course,
                            "chunk_index": abs_idx,
                            "mode": mode,
                            "n_questions": n_questions,
                            "input": chunk,
                            "output": output,
                        })

                    done_keys.add(key)
                    pbar.update(1)
                    save_progress(final_dataset, done_keys, failed_samples)

    print(f"\nDone (this cell). Valid samples so far : {len(final_dataset)}")
    print(f"Failed samples so far                 : {failed_samples}")
    return final_dataset, done_keys, failed_samples


## Generate — **GIS** (Part A, chunks 0–89)

90 chunks × 3 modes = 270 calls

In [ ]:
# GIS — Part A  (chunks 0 to 89)
run_generation_for_courses([
    ('GIS', courses_data['GIS'], 0, 90),
])


## Generate — **GIS** (Part B, chunks 90–272)

183 chunks × 3 modes 

In [ ]:
# GIS — Part B  (chunks 90 to 272)
run_generation_for_courses([
    ('GIS', courses_data['GIS'], 90, 273),
])


In [ ]:
# Cloud_Computing — Part A  (chunks 0 to 89)
run_generation_for_courses([
    ('Cloud_Computing', courses_data['Cloud_Computing'], 0, 90),
])


In [ ]:
# Cloud_Computing — Part B  (chunks 90 to 231)
run_generation_for_courses([
    ('Cloud_Computing', courses_data['Cloud_Computing'], 90, 232),
])


In [ ]:
# Software_Quality_Assurance — Part A  (chunks 0 to 89)
run_generation_for_courses([
    ('Software_Quality_Assurance', courses_data['Software_Quality_Assurance'], 0, 90),
])


In [ ]:
# Software_Quality_Assurance — Part B  (chunks 90 to 230)
run_generation_for_courses([
    ('Software_Quality_Assurance', courses_data['Software_Quality_Assurance'], 90, 231),
])


In [ ]:
# Compilers_Theory — Part A  (chunks 0 to 89)
run_generation_for_courses([
    ('Compilers_Theory', courses_data['Compilers_Theory'], 0, 90),
])


In [ ]:
# Compilers_Theory — Part B  (chunks 90 to 184)
run_generation_for_courses([
    ('Compilers_Theory', courses_data['Compilers_Theory'], 90, 185),
])


In [ ]:
# HCI — Part A  (chunks 0 to 89)
run_generation_for_courses([
    ('HCI', courses_data['HCI'], 0, 90),
])


In [ ]:
# HCI — Part B  (chunks 90 to 179)
run_generation_for_courses([
    ('HCI', courses_data['HCI'], 90, 180),
])


In [ ]:
# cyber_security — Part A  (chunks 0 to 89)
run_generation_for_courses([
    ('cyber_security', courses_data['cyber_security'], 0, 90),
])


In [ ]:
# cyber_security — Part B  (chunks 90 to 176)
run_generation_for_courses([
    ('cyber_security', courses_data['cyber_security'], 90, 177),
])


In [ ]:
# neural_networks — Part A  (chunks 0 to 89)
run_generation_for_courses([
    ('neural_networks', courses_data['neural_networks'], 0, 90),
])


In [ ]:
# neural_networks — Part B  (chunks 90 to 174)
run_generation_for_courses([
    ('neural_networks', courses_data['neural_networks'], 90, 175),
])


In [ ]:
# data_science — Part A  (chunks 0 to 89)
run_generation_for_courses([
    ('data_science', courses_data['data_science'], 0, 90),
])


In [ ]:
# data_science — Part B  (chunks 90 to 170)
run_generation_for_courses([
    ('data_science', courses_data['data_science'], 90, 171),
])


In [ ]:
# NLP — Part A  (chunks 0 to 89)
run_generation_for_courses([
    ('NLP', courses_data['NLP'], 0, 90),
])


In [ ]:
# NLP — Part B  (chunks 90 to 151)
run_generation_for_courses([
    ('NLP', courses_data['NLP'], 90, 152),
])


In [ ]:
# LP — Part A  (chunks 0 to 89)
run_generation_for_courses([
    ('LP', courses_data['LP'], 0, 90),
])


In [ ]:
# LP — Part B  (chunks 90 to 145)
run_generation_for_courses([
    ('LP', courses_data['LP'], 90, 146),
])


In [ ]:
# Statistical_Inference — Part A  (chunks 0 to 89)
run_generation_for_courses([
    ('Statistical_Inference', courses_data['Statistical_Inference'], 0, 90),
])


In [ ]:
# Statistical_Inference — Part B  (chunks 90 to 120)
run_generation_for_courses([
    ('Statistical_Inference', courses_data['Statistical_Inference'], 90, 121),
])


In [ ]:
# Distributed — Part A  (chunks 0 to 89)
run_generation_for_courses([
    ('Distributed', courses_data['Distributed'], 0, 90),
])


In [ ]:
# Distributed — Part B  (chunks 90 to 116)
run_generation_for_courses([
    ('Distributed', courses_data['Distributed'], 90, 117),
])


In [ ]:
# Social_Network_Analysis — Part A  (chunks 0 to 89)
run_generation_for_courses([
    ('Social_Network_Analysis', courses_data['Social_Network_Analysis'], 0, 90),
])


In [ ]:
# Social_Network_Analysis — Part B  (chunks 90 to 110)
run_generation_for_courses([
    ('Social_Network_Analysis', courses_data['Social_Network_Analysis'], 90, 111),
])


In [ ]:
# Theory_of_Computation — Part A  (chunks 0 to 89)
run_generation_for_courses([
    ('Theory_of_Computation', courses_data['Theory_of_Computation'], 0, 90),
])


In [ ]:
# Theory_of_Computation — Part B  (chunks 90 to 99)
run_generation_for_courses([
    ('Theory_of_Computation', courses_data['Theory_of_Computation'], 90, 100),
])


In [ ]:
# software — Part A  (chunks 0 to 89)
run_generation_for_courses([
    ('software', courses_data['software'], 0, 90),
])


In [ ]:
# software — Part B  (chunks 90 to 98)
run_generation_for_courses([
    ('software', courses_data['software'], 90, 99),
])


In [ ]:
# system_analysis — Part A  (chunks 0 to 89)
run_generation_for_courses([
    ('system_analysis', courses_data['system_analysis'], 0, 90),
])


In [ ]:
# system_analysis — Part B  (chunks 90 to 94)
run_generation_for_courses([
    ('system_analysis', courses_data['system_analysis'], 90, 95),
])


In [ ]:
# network
run_generation_for_courses([
    ('network', courses_data['network']),
])


In [ ]:
# IOT
run_generation_for_courses([
    ('IOT', courses_data['IOT']),
])


In [ ]:
# DSP
run_generation_for_courses([
    ('DSP', courses_data['DSP']),
])


In [ ]:
# Computational_Geometry
run_generation_for_courses([
    ('Computational_Geometry', courses_data['Computational_Geometry']),
])
